<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/InceptionV3%2BRF(Bayesian).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer

import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

In [ ]:
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_lab = np.load('/content/drive/MyDrive/Y_train_labels.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_lab = np.load('/content/drive/MyDrive/Y_test_labels.npy')

In [ ]:
print("--- LABEL MAPPING KEY ---")

class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    60: ("Bare / Sparse vegetation", "#b4b4b4"),
    70: ("Snow and ice", "#f0f0f0"),
    80: ("Permanent water bodies", "#0064ff"),
    90: ("Herbaceous wetland", "#0096a0"),
}

# Create the internal mapping
unique_labels = sorted(np.unique(Y_train_lab))
label_map = {old: new for new, old in enumerate(unique_labels)}

# Sorting by the new index (0, 1, 2...) for readability
for old_id, new_id in sorted(label_map.items(), key=lambda item: item[1]):
    class_name = class_map.get(old_id, ("Unknown", ""))[0]
    print(f"New ID: {new_id}  <--  Original ID: {old_id} ({class_name})")

# Apply the mapping to create the final training/testing labels
Y_train_ready = np.array([label_map[l] for l in Y_train_lab])
Y_test_ready = np.array([label_map[l] for l in Y_test_lab])


print("\n--- DATA SHAPE VERIFICATION ---")
print(f"X_train: {X_train.shape}")
print(f"Y_train_ready: {Y_train_ready.shape}")
print(f"Unique classes in Training: {np.unique(Y_train_ready)}")

--- LABEL MAPPING KEY ---
New ID: 0  <--  Original ID: 10 (Tree cover)
New ID: 1  <--  Original ID: 20 (Shrubland)
New ID: 2  <--  Original ID: 30 (Grassland)
New ID: 3  <--  Original ID: 40 (Cropland)
New ID: 4  <--  Original ID: 50 (Built-up)
New ID: 5  <--  Original ID: 80 (Permanent water bodies)

--- DATA SHAPE VERIFICATION ---
X_train: (639, 256, 256, 7)
Y_train_ready: (639,)
Unique classes in Training: [0 1 2 3 4 5]


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np

# Data Augmentation (Safe Flips & Brightness)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomBrightness(0.1),
])

# Model Input
inputs = Input(shape=(256, 256, 7))
x = data_augmentation(inputs)

# Adapter
adapter = Conv2D(3, (1, 1), padding='same', name='band_adapter')(x)

# InceptionV3 Backbone
inception_base = tf.keras.applications.InceptionV3(
    include_top=False,
    weights='imagenet',
    input_shape=(256, 256, 3)
)

# Connect and Pool
x = inception_base(adapter)
pooled_output = GlobalAveragePooling2D(name='feature_layer')(x)


# Head for Fine-Tuning
x_head = Dense(256, activation='relu')(pooled_output)
x_head = Dropout(0.5)(x_head)
temp_predictions = Dense(len(np.unique(Y_train_ready)), activation='softmax')(x_head)

# Model for Training (Phase 1 & 2)
trainable_model = Model(inputs=inputs, outputs=temp_predictions)

# Model for Feature Extraction
feature_extractor = Model(inputs=inputs, outputs=pooled_output)

print("Feature Extractor Defined Successfully.")

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Feature Extractor Defined Successfully.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import time
import psutil
import torch

# Calculate weights for class balance

classes = np.unique(Y_train_ready)
weights = compute_class_weight('balanced', classes=classes, y=Y_train_ready)
class_weight_dict = dict(enumerate(weights))

# Phase 1: Freeze Inception, Train Adapter and Head
inception_base.trainable = False

trainable_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])

process = psutil.Process()
torch.cuda.reset_peak_memory_stats()
start_cnn = time.time()

print("Starting Phase 1: Warming up Adapter and Head...")
trainable_model.fit(X_train, Y_train_ready,
                    epochs=5,
                    batch_size=16,
                    validation_data=(X_test, Y_test_ready),
                    class_weight=class_weight_dict,
                    verbose=1
                    )

end_cnn = time.time()
print(f"CNN Training Time: {end_cnn - start_cnn:.2f} seconds")
print(f"System RAM Usage:  {process.memory_info().rss / 1024**2:.2f} MB")

Starting Phase 1: Warming up Adapter and Head...
Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 66s 970ms/step - accuracy: 0.1628 - loss: 2.1456 - val_accuracy: 0.2400 - val_loss: 1.8077
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 108ms/step - accuracy: 0.1659 - loss: 1.9379 - val_accuracy: 0.3120 - val_loss: 1.7227
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - accuracy: 0.1925 - loss: 1.8810 - val_accuracy: 0.4160 - val_loss: 1.6333
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 111ms/step - accuracy: 0.1674 - loss: 1.8670 - val_accuracy: 0.3200 - val_loss: 1.6796
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 107ms/step - accuracy: 0.1596 - loss: 1.8338 - val_accuracy: 0.3200 - val_loss: 1.6923
CNN Training Time: 87.81 seconds
System RAM Usage:  5536.88 MB


In [ ]:
# Phase 2: Fine-Tuning
inception_base.trainable = True

# Lower Learning Rate to nudge weights specifically for satellite textures
trainable_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

process = psutil.Process()
torch.cuda.reset_peak_memory_stats()
start_cnn = time.time()

print("\nStarting Phase 2: Fine-tuning Inception for Satellite Data...")
trainable_model.fit(X_train, Y_train_ready,
                    epochs=10,
                    batch_size=16,
                    validation_data=(X_test, Y_test_ready),
                    class_weight=class_weight_dict,
                    verbose=1
                    )

end_cnn = time.time()
print(f"CNN Training Time: {end_cnn - start_cnn:.2f} seconds")
print(f"System RAM Usage:  {process.memory_info().rss / 1024**2:.2f} MB")


Starting Phase 2: Fine-tuning Inception for Satellite Data...
Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - accuracy: 0.1972 - loss: 1.7899 - val_accuracy: 0.3200 - val_loss: 1.7013
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 184ms/step - accuracy: 0.2504 - loss: 1.7208 - val_accuracy: 0.3680 - val_loss: 1.6613
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 192ms/step - accuracy: 0.2989 - loss: 1.6511 - val_accuracy: 0.4800 - val_loss: 1.5460
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 192ms/step - accuracy: 0.3192 - loss: 1.5710 - val_accuracy: 0.5360 - val_loss: 1.3884
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 189ms/step - accuracy: 0.3599 - loss: 1.4896 - val_accuracy: 0.6240 - val_loss: 1.1397
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 196ms/step - accuracy: 0.3631 - loss: 1.4746 - val_accuracy: 0.6800 - val_loss: 1.0058
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 192ms/step - accuracy: 0.4147 - loss: 1.3887 - val_accuracy: 0.7040 - val_loss: 0.9382
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

print("Extracting Raw Spectral Means...")
# Reduces (N, 256, 256, 7) to (N, 7)
X_train_raw_means = np.mean(X_train, axis=(1, 2))
X_test_raw_means = np.mean(X_test, axis=(1, 2))

print("Extracting Inception Features...")
X_train_inc = feature_extractor.predict(X_train, batch_size=16)
X_test_inc = feature_extractor.predict(X_test, batch_size=16)

print("Applying PCA (Reducing 2048 -> 100 features)...")
# This prevents the Inception features from overpowering the spectral bands
pca = PCA(n_components=100, random_state=20)
X_train_pca = pca.fit_transform(X_train_inc)
X_test_pca = pca.transform(X_test_inc)

print("Fusing Features...")
# Combine the 100 shape-based features with the 7 color-based features
X_train_final = np.hstack((X_train_pca, X_train_raw_means))
X_test_final = np.hstack((X_test_pca, X_test_raw_means))

print(f"\nFinal Hybrid Dataset Ready!")
print(f"Train Shape: {X_train_final.shape}") # Should be (N, 107)
print(f"Test Shape:  {X_test_final.shape}")

Extracting Raw Spectral Means...
Extracting Inception Features...
40/40 ━━━━━━━━━━━━━━━━━━━━ 17s 246ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step
Applying PCA (Reducing 2048 -> 100 features)...
Fusing Features...

Final Hybrid Dataset Ready!
Train Shape: (639, 107)
Test Shape:  (125, 107)


In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real
from sklearn.ensemble import RandomForestClassifier

search_space = {
    'n_estimators': Integer(200, 600),
    'max_depth': Integer(5, 20),
    'min_samples_split': Integer(10, 50),
    'max_features': Categorical(['sqrt', 'log2']),
    'max_samples': Real(0.5, 0.8)
}

rf = RandomForestClassifier(random_state=20, class_weight='balanced', n_jobs=-1)

opt = BayesSearchCV(
    estimator=rf,
    search_spaces=search_space,
    n_iter=15,
    cv=3,
    n_jobs=-1,
    verbose=1,
    random_state=20
)
start_rf = time.time()

opt.fit(X_train_final, Y_train_ready)

end_rf = time.time()
print(f"RF Optimization Time: {end_rf - start_rf:.2f} seconds")
print(f"System RAM Usage:     {process.memory_info().rss / 1024**2:.2f} MB")

print(f"\nBest Parameters found: {opt.best_params_}")
print(f"Best Validation F1-Macro: {opt.best_score_:.4f}")


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
RF Optimization Time: 75.49 seconds
System RAM Usage:     7128.87 MB

Best Parameters found: OrderedDict({'max_dept

In [ ]:
from sklearn.metrics import cohen_kappa_score, jaccard_score

best_rf = opt.best_estimator_
y_pred = best_rf.predict(X_test_final)

target_names = ['Tree cover', 'Shrubland', 'Grassland', 'Cropland', 'Built-up', 'Permanent water']

# Final predictions
train_preds = best_rf.predict(X_train_final)
test_preds = best_rf.predict(X_test_final)

# Calculate metrics
train_acc = accuracy_score(Y_train_ready, train_preds)
test_acc = accuracy_score(Y_test_ready, test_preds)
test_f1 = f1_score(Y_test_ready, test_preds, average='macro')
# Calculate Kappa
kappa = cohen_kappa_score(Y_test_ready, y_pred)
# Calculate Macro IoU (Jaccard Score)
iou = jaccard_score(Y_test_ready, y_pred, average='macro')

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")
print(f"Mean IoU:      {iou:.4f}")

print(classification_report(Y_test_ready, y_pred, target_names=target_names))

Train Accuracy: 0.9906
Test Accuracy:  0.8320
Test Macro F1:  0.8110
Cohen's Kappa: 0.7936
Mean IoU:      0.6932
                 precision    recall  f1-score   support

     Tree cover       0.60      1.00      0.75         6
      Shrubland       0.71      0.77      0.74        13
      Grassland       0.89      0.59      0.71        29
       Cropland       0.90      0.93      0.91        28
       Built-up       0.73      0.86      0.79        22
Permanent water       0.96      0.96      0.96        27

       accuracy                           0.83       125
      macro avg       0.80      0.85      0.81       125
   weighted avg       0.85      0.83      0.83       125

